# Requirement 10: Session-specific authorization

> Every sensitive interpreter operation must require session-specific authentication. Network reachability or Azure permissions alone must not grant access to session contents.

**Covered by:** Caller

**How it is fulfilled:** Each operation carries an HMAC token derived from the per-session key and bound to the session id, a counter (replay protection), and the exact request body. A handler in the sandbox rejects anything else.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
sandbox, session_id = new_session()
session_key = install_session_key(sandbox, session_id)

In [3]:
good = signed_request(session_key, session_id, 1, [1, 2, 3])
out = send(sandbox, good)
print("valid request          ->", out)
check("valid request accepted", out.startswith("OK 6"))
out = send(sandbox, good)
print("replay of that request ->", out)
check("replay rejected", "REJECTED: replay" in out)

valid request          -> OK 6 exit=0
PASS  valid request accepted


replay of that request -> REJECTED: replay exit=3
PASS  replay rejected


In [4]:
tampered = signed_request(session_key, session_id, 2, [1, 2, 3])
tampered["body"]["numbers"] = [1000]
check("tampered body rejected", "REJECTED" in send(sandbox, tampered))
check("missing token rejected", "REJECTED" in send(sandbox, {"session": session_id, "counter": 50, "body": {"numbers": [1]}}))
check("wrong session id rejected", "REJECTED: wrong session" in send(sandbox, signed_request(session_key, "someone-else", 60, [1])))
check("wrong key rejected", "REJECTED" in send(sandbox, signed_request(os.urandom(32), session_id, 70, [1])))

PASS  tampered body rejected


PASS  missing token rejected


PASS  wrong session id rejected


PASS  wrong key rejected


In [5]:
# The limit: raw exec by a Data Owner holder skips the handler
print("key bytes readable via raw exec:", sandbox.exec("wc -c < /tmp/session_key.bin").stdout.strip())
cleanup(sandbox)

key bytes readable via raw exec: 32


**Caveat:** This binds operations to one session and one caller. It cannot stop a Data Owner holder, who can use raw exec or file access. The real gate is that only the orchestrator holds the role.